# Svarati: Pipeline RAG (Single Model intfloat/multilingual-e5-base)

Notebook ini diadaptasi khusus untuk project **Svarati (Virtual Safe Space)**.

### Fitur & Pembaruan:
1. **Single Model Execution**: Menggunakan model embedding `intfloat/multilingual-e5-base` (768 dimensi, mendukung hingga 512 token).
2. **Tanpa Perbandingan Model Otomatis**: Tidak ada loop perbandingan model di dalam kode. Untuk membandingkan dengan model lain (mis. `BAAI/bge-m3`), ganti `MODEL_NAME` lalu jalankan ulang.
3. **Cek Kondisi (Q1–Q6)**: Jawaban self-check diubah menjadi `topics` dan `prefer_types` untuk memperkuat retrieval (lihat Alur 2).
4. **Gerbang Keselamatan & Psychological First Aid (PFA)**: Alur RAG dihentikan dan kontak darurat (112 / 129 SAPA KemenPPPA) langsung ditampilkan jika `q1 == 'a'` atau pesan bebas memuat tanda bahaya.
5. **Cek Kutipan Pasal**: Memastikan pasal yang disebut Gemini benar-benar ada di hasil retrieval.


## 1. Latar Belakang Masalah

Banyak perempuan di Indonesia menghadapi keterbatasan akses terhadap pemahaman hukum, hak-hak perlindungan diri, serta prosedur penanganan kasus kekerasan/pelecehan secara cepat dan akurat. Bahasa hukum yang rumit dan informasi yang tersebar kerap menjadi hambatan utama saat korban atau pendamping membutuhkan rujukan darurat.

**Pengguna Sasaran:** Perempuan, korban kekerasan/pelecehan, aktivis/pendamping lapangan, serta masyarakat umum yang membutuhkan informasi hukum praktis dan cepat.

## 2. Deskripsi Fitur

**Svarati AI** adalah satu asisten yang menjawab dengan empati, memberi penjelasan hukum, dan menyusun langkah-langkah yang bisa diikuti. Semuanya dijalankan oleh satu model dan satu prompt, bukan tiga AI terpisah. Pada versi notebook ini, jenis jawaban belum dipilih secara eksplisit di kode; isi jawaban mengikuti pertanyaan pengguna dan dokumen yang ditemukan. Pemilihan jenis jawaban eksplisit termasuk pengembangan lanjutan.

**Parameter RAG:** selain pertanyaan pengguna, Svarati AI menerima hasil **self-check Q1–Q6** (Ya/Tidak). Hasilnya dipakai untuk memperkuat pencarian pasal dan dimasukkan ke query retrieval (lihat Alur 2 dan bagian 8).

**Peran di final project** (integrasi backend, di luar notebook ini): halaman Svarati AI yang menjawab dengan empati dan merujuk pasal (RAG). Percakapan disimpan di tabel `ai_messages`beserta sumber yang dikutip di `ai_message_sources`, dan status validasi kasus disimpan di `anonymous_sessions.validated_category`; jawaban per pertanyaan tidak disimpan.

**Prinsip jawaban** (sama dengan `SYSTEM_PROMPT`):
1. Hangat, empatik, dan tidak menyalahkan korban.
2. Jawaban hanya berdasarkan konteks dokumen.
3. Tidak mengarang pasal, nomor telepon, atau alamat. Hanya merujuk pasal yang benar-benar ditemukan.
4. Menghormati keputusan pengguna; pilihan langkah diberikan tanpa tekanan.
5. Jika konteks tidak mencukupi, sampaikan keterbatasannya dengan jujur dan sarankan layanan terverifikasi.
6. Tidak mendiagnosis dan tidak menjanjikan hasil hukum.
7. Jika ada tanda bahaya, arahkan ke kontak darurat dan layanan profesional.


## 3. Alur Pipeline

Pipeline Svarati AI terdiri dari dua alur: **ingestion dokumen** yang dijalankan sekali di awal (offline), dan **tanya-jawab** yang dijalankan setiap kali pengguna bertanya (online).

### Alur 1: Ingestion Dokumen (offline)

```
Sumber Data → Ekstraksi → Pembersihan & Chunking → Metadata Topik → Embedding → Index Vektor
```

| Tahap | Penjelasan | Kode di notebook |
|---|---|---|
| 1. Sumber data | PDF/Google Docs dari Google Drive (saat ini UU TPKS) dan data fiktif untuk uji lokal | `DRIVE_SOURCES`, `DUMMY_DOCS` |
| 2. Ekstraksi | Mengunduh file dari Drive dan mengambil teks PDF dengan `pypdf` | `drive_id()`, `fetch_text()` |
| 3. Pembersihan & chunking | Merapikan spasi/baris kosong. Dokumen hukum dipecah per **Pasal**, lalu pasal yang melebihi ukuran chunk dipecah lagi. Dokumen lain dipecah per 1000 karakter dengan overlap 180 | `bersihkan()`, `pecah_hukum()`, `pecah_panjang()` |
| 4. Metadata topik | Setiap chunk diberi `topic` otomatis berdasarkan kata kunci (mis. `keamanan_diri`, `hukum_tpks`) | `TOPIC_KEYWORDS`, `auto_topic()`, `bangun_entries()` |
| 5. Embedding | Chunk diubah menjadi vektor ternormalisasi dengan `intfloat/multilingual-e5-base` (awalan `passage:`) | `Embedder.docs()` |
| 6. Index vektor | Matriks vektor disimpan di memori bersama metadata | `Index`, `idx` |

> **Catatan keterbatasan:** `DRIVE_SOURCES` saat ini hanya berisi UU TPKS, sehingga index belum memuat chunk bertipe `prosedur` atau `panduan_dukungan`. Akibatnya `TYPE_BOOST` dan `prefer_types` belum berpengaruh, dan jawaban bisa menyatakan dokumen tidak memuat panduan (misalnya cara menyimpan bukti). Tambahkan sumber panduan ke `DRIVE_SOURCES` untuk mengaktifkannya.

### Alur 2: Tanya-Jawab (online)

```
Pengguna → Gerbang Keselamatan → Rencana RAG → Susun Query → Retrieval → Prompt → Generation (Gemini) → Cek Kutipan
```

| Tahap | Penjelasan | Kode di notebook |
|---|---|---|
| 1. Input pengguna | Pesan bebas + jawaban **Cek Kondisi (Q1–Q6)** | `Q1` ... `Q6` |
| 2. Gerbang keselamatan | Alur RAG dihentikan dan langsung menampilkan kontak darurat **112 / 129 (SAPA KemenPPPA)** jika `q1 == "a"` (dalam bahaya) atau pesan bebas memuat tanda bahaya | `gerbang()`, `deteksi_darurat()` |
| 3. Rencana RAG | Jawaban Q1–Q6 diubah menjadi daftar `topics` dan `prefer_types` untuk memperkuat pencarian | `rencana()`, `MAP_Q2/Q3/Q4` |
| 4. Susun query | Pesan pengguna digabung dengan konteks kejadian, lokasi, pihak terlibat, dan bukti. Hasilnya dipakai sebagai query retrieval dan sebagai bagian pertanyaan di prompt | `susun_query()` |
| 5. Retrieval | Query di-embed (awalan `query:`), dihitung kemiripan kosinus, ditambah bonus topik (+0.05) dan tipe (+0.03), disaring `MIN_SIM = 0.30`, lalu diambil `TOP_K = 3` | `cari()` |
| 6. Prompt | Hasil retrieval disusun sebagai `SUMBER`, lalu dikirim bersama query hasil `susun_query()` dan `SYSTEM_PROMPT` (pedoman PFA dan prinsip jawaban) | `bangun_prompt()`, `SYSTEM_PROMPT` |
| 7. Generation | Gemini menjawab hanya berdasarkan konteks dokumen (temperature 0.2) | `tanya_svarati()` |
| 8. Cek kutipan | Memastikan pasal yang disebut AI benar-benar ada di hasil retrieval | `cek_kutipan()` |

**Mode keluaran** `tanya_svarati()`:
- `darurat`: gerbang keselamatan aktif, langsung menampilkan kontak darurat.
- `tanpa_sumber`: tidak ada chunk yang lolos `MIN_SIM`. Sistem mengembalikan satu pesan standar yang mengarahkan pengguna ke pendamping atau layanan terverifikasi, tanpa memanggil LLM.
- `tanpa_llm`: API key Gemini belum diisi, hanya retrieval.
- `rag`: jawaban lengkap dari Gemini beserta pengecekan pasal.


In [13]:
!pip -q install sentence-transformers pypdf pandas numpy requests google-genai

## 4. Konfigurasi Model & Parameter RAG

In [14]:
import os, re, io, json
import numpy as np
import pandas as pd

# ---- Model embedding utama ------------------------------------------------------
MODEL_NAME = "intfloat/multilingual-e5-base"

# ---- Parameter Chunking & Retrieval --------------------------------------------
CHUNK_SIZE, CHUNK_OVERLAP = 1000, 180
TOP_K = 3
MIN_SIM = 0.30        # Ambang batas kemiripan kosinus minimal
TOPIC_BOOST = 0.05    # Bonus skor jika topic cocok dengan hasil Cek Kondisi
TYPE_BOOST = 0.03     # Bonus skor jika entry_type cocok

# ---- Sumber data Google Drive (Opsional) --------------------------------------
# Isi list berikut jika ingin menarik PDF/Google Docs langsung dari Google Drive.
DRIVE_SOURCES = [
    {"entry_type": "hukum", "source_name": "UU No. 12 Tahun 2022 (TPKS)",
     "title": "UU TPKS", "url": "https://drive.google.com/file/d/1_F-3XvAzCFFSXAsRb8_rQQIc3XQU7Qw3/view?usp=sharing"},
]

## 5. Helper Pengambil Dokumen dari Google Drive

In [15]:
def drive_id(url):
    m = re.search(r"/d/([a-zA-Z0-9_-]+)", url) or re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
    if not m:
        raise ValueError(f"Tidak menemukan ID file di link: {url}")
    return m.group(1)

def fetch_text(url):
    import requests
    fid = drive_id(url)
    if "docs.google.com/document" in url:                      # Google Docs -> txt
        r = requests.get(f"https://docs.google.com/document/d/{fid}/export?format=txt", timeout=60)
        r.raise_for_status()
        return r.content.decode("utf-8", errors="ignore")
    r = requests.get(f"https://drive.google.com/uc?export=download&id={fid}", timeout=60)
    r.raise_for_status()
    data = r.content
    if data[:4] == b"%PDF":                                      # PDF
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(data))
        return "\n".join((p.extract_text() or "") for p in reader.pages)
    if b"<html" in data[:2000].lower():
        raise RuntimeError("Drive mengembalikan halaman HTML. Pastikan berbagi 'Siapa saja yang memiliki link'.")
    return data.decode("utf-8", errors="ignore")

## 6. Pemecahan Dokumen (Chunking) & Metadata Topik

In [16]:
# Topik disesuaikan dengan enum database Svarati
TOPIC_KEYWORDS = {
    "keamanan_diri":         ["elektronik", "digital", "daring", "online", "media sosial", "foto", "video", "rekaman", "ancaman"],
    "pelecehan_nonfisik":    ["nonfisik", "non-fisik", "komentar", "candaan", "isyarat", "ucapan"],
    "pelecehan_fisik":       ["fisik", "sentuh", "meraba", "cabul"],
    "pemaksaan_hubungan":    ["pemaksaan", "perkosaan", "persetubuhan"],
    "hukum_tpks":            ["uu tpks", "pasal", "tindak pidana", "sanksi", "hakim", "penyidik"],
    "hak_perempuan":         ["hak korban", "restitusi", "pemulihan", "perlindungan", "pendampingan"],
    "kesehatan_mental":      ["trauma", "cemas", "takut", "bingung", "psikolog", "konseling", "pfa"],
    "kesehatan_reproduksi":   ["pemeriksaan medis", "fisik", "visum", "penyakit menular", "kehamilan"]
}

def auto_topic(teks):
    t = teks.lower()
    skor = {
        k: sum(len(re.findall(r"(?<![\w-])" + re.escape(w) + r"(?![\w-])", t)) for w in ws)
        for k, ws in TOPIC_KEYWORDS.items()
    }
    k, v = max(skor.items(), key=lambda kv: kv[1])
    return k if v > 0 else "hak_perempuan"

def bersihkan(t):
    t = t.replace("\r", "")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def pecah_panjang(teks, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    teks = teks.strip()
    if len(teks) <= size:
        return [teks] if teks else []
    out, mulai = [], 0
    while mulai < len(teks):
        akhir = min(mulai + size, len(teks))
        if akhir < len(teks):
            cut = max(teks.rfind("\n", mulai, akhir), teks.rfind(" ", mulai, akhir))
            if cut > mulai + size // 2:
                akhir = cut
        out.append(teks[mulai:akhir].strip())
        if akhir >= len(teks):
            break
        mulai = max(akhir - overlap, mulai + 1)
    return [c for c in out if c]

PASAL_RE = re.compile(r"(?m)^\s*Pasal\s+(\d+[A-Z]?)\s*$")

def pecah_hukum(teks):
    ms = list(PASAL_RE.finditer(teks))
    if not ms:
        return [(None, c) for c in pecah_panjang(teks)]
    hasil = []
    for i, m in enumerate(ms):
        akhir = ms[i + 1].start() if i + 1 < len(ms) else len(teks)
        hasil.append((m.group(1), teks[m.start():akhir].strip()))
    return hasil

def bangun_entries(sumber, teks):
    teks = bersihkan(teks)
    rows = []
    if sumber["entry_type"] == "hukum":
        for nomor, isi in pecah_hukum(teks):
            bagian = pecah_panjang(isi)
            for j, c in enumerate(bagian, 1):
                judul = f"{sumber['title']} - Pasal {nomor}" if nomor else f"{sumber['title']} (bagian {len(rows)+1})"
                if len(bagian) > 1:
                    judul += f" (bagian {j})"
                rows.append(dict(entry_type="hukum", source_name=sumber["source_name"], title=judul,
                                 article_number=nomor, original_text=c, simplified_text=None,
                                 topic=sumber.get("topic") or auto_topic(c),
                                 source_url=sumber.get("url"), is_active=True))
    else:
        for j, c in enumerate(pecah_panjang(teks), 1):
            rows.append(dict(entry_type=sumber["entry_type"], source_name=sumber["source_name"],
                             title=f"{sumber['title']} (bagian {j})", article_number=None,
                             original_text=c, simplified_text=None,
                             topic=sumber.get("topic") or auto_topic(c),
                             source_url=sumber.get("url"), is_active=True))
    return rows

In [17]:
# Data Fiktif untuk Pengujian Lokal
DUMMY_DOCS = [
    ({"entry_type": "hukum", "source_name": "[CONTOH] UU TPKS Fiktif", "title": "UU TPKS (Kekerasan Seksual Berbasis Elektronik)", "url": "https://contoh.invalid/tpks"},
     "Pasal 14\n(1) Setiap Orang yang melakukan perekaman dan/atau transmisi konten bermuatan seksual tanpa persetujuan dipidana karena kekerasan seksual berbasis elektronik.\n"
     "Pasal 57\nPenuntut umum wajib menjamin perlindungan identitas dan kerahasiaan dokumen pribadi korban kekerasan seksual."),
    ({"entry_type": "panduan_dukungan", "source_name": "[CONTOH] Panduan PFA Svarati", "title": "Panduan Psychological First Aid", "url": "https://contoh.invalid/pfa"},
     "Membeku, bingung, atau cemas setelah menghadapi kejadian kekerasan adalah respon traumatis yang wajar. "
     "Dengarkan tanpa menghakimi, validasi perasaan korban, dan berikan opsi bantuan profesional tanpa memaksakan keputusan."),
    ({"entry_type": "prosedur", "source_name": "[CONTOH] Panduan Bukti Digital", "title": "Prosedur Penyimpanan Bukti", "url": "https://contoh.invalid/bukti"},
     "Ambil tangkapan layar (screenshot) percakapan, catat tanggal dan waktu, simpan pesan asli, dan buat cadangan aman di media penyimpanan lain.")
]

def muat_entries():
    rows = []
    if DRIVE_SOURCES:
        for s in DRIVE_SOURCES:
            print("Mengambil dari Drive:", s["title"])
            rows += bangun_entries(s, fetch_text(s["url"]))
    else:
        print("DRIVE_SOURCES kosong -> Memakai DATA CONTOH FIKTIF Svarati")
        for s, teks in DUMMY_DOCS:
            rows += bangun_entries(s, teks)
    return pd.DataFrame(rows)

entries = muat_entries()
print(f"Total potongan dokumen terproses: {len(entries)}")
entries[["entry_type", "title", "article_number", "topic"]].head(10)

Mengambil dari Drive: UU TPKS
Total potongan dokumen terproses: 207


,entry_type,title,article_number,topic
0,hukum,UU TPKS - Pasal 1 (bagian 1),1,hukum_tpks
1,hukum,UU TPKS - Pasal 1 (bagian 2),1,hukum_tpks
2,hukum,UU TPKS - Pasal 1 (bagian 3),1,hak_perempuan
3,hukum,UU TPKS - Pasal 1 (bagian 4),1,hak_perempuan
4,hukum,UU TPKS - Pasal 1 (bagian 5),1,hak_perempuan
5,hukum,UU TPKS - Pasal 1 (bagian 6),1,hak_perempuan
6,hukum,UU TPKS - Pasal 1 (bagian 7),1,hak_perempuan
7,hukum,UU TPKS - Pasal 2,2,hukum_tpks
8,hukum,UU TPKS - Pasal 3,3,hukum_tpks
9,hukum,UU TPKS - Pasal 4 (bagian 1),4,pemaksaan_hubungan


## 7. Embedder & Index Vektor (`intfloat/multilingual-e5-base`)

In [18]:
class Embedder:
    def __init__(self, name):
        from sentence_transformers import SentenceTransformer
        self.name = name
        self.is_e5 = "e5" in name.lower()
        self.model = SentenceTransformer(name)
        self.model.max_seq_length = 512 if self.is_e5 else 768

    def _enc(self, texts, bs=16):
        v = self.model.encode(list(texts), batch_size=bs, normalize_embeddings=True, show_progress_bar=False)
        return np.asarray(v, dtype="float32")

    def docs(self, texts):
        return self._enc([("passage: " if self.is_e5 else "") + t for t in texts])

    def queries(self, texts):
        return self._enc([("query: " if self.is_e5 else "") + t for t in texts])

class Index:
    def __init__(self, model_name, entries_df):
        self.emb = Embedder(model_name)
        self.kb = entries_df.reset_index(drop=True).copy()
        teks = (self.kb["title"] + "\n" + self.kb["original_text"]).tolist()
        self.matrix = self.emb.docs(teks)
        self.kb["embedding_model"] = model_name
        self.dim = self.matrix.shape[1]

idx = Index(MODEL_NAME, entries)
print(f"Model Aktif: {MODEL_NAME} | Dimensi Vector: {idx.dim} | Jumlah Chunk: {len(idx.kb)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model Aktif: intfloat/multilingual-e5-base | Dimensi Vector: 768 | Jumlah Chunk: 207


## 8. Cek Kondisi (Q1–Q6): Gerbang Keselamatan, Rencana RAG & Retrieval

In [19]:
Q1 = {"a": "Dalam bahaya (mengancam jiwa atau kesehatan)", "b": "Aman, tapi bingung", "c": "Aman, tapi masih trauma",
      "d": "Aman, hanya ingin tahu soal hukum atau hak perempuan"}
Q2 = {"a": "Online (chat, medsos)", "b": "Kampus, sekolah, magang, kos", "c": "Kantor atau tempat kerja",
      "d": "Tempat umum", "e": "Rumah atau keluarga"}
Q3 = {"a": "Atasan atau dosen", "b": "Teman atau rekan", "c": "Pacar, mantan, atau keluarga", "d": "Orang yang baru dikenal"}
Q4 = {"a": "ada candaan atau komentar bernuansa seksual", "b": "dikomentari soal tubuh",
      "c": "dikirimi foto atau video tanpa izin", "d": "diancam foto atau datanya disebar",
      "e": "diintip atau direkam diam-diam", "f": "disentuh tanpa izin", "g": "dipaksa berhubungan intim",
      "h": "nilai atau kerja dikaitkan dengan permintaan intim"}
Q5 = {"a": "sudah menolak, tapi tetap dilanjutkan", "b": "membeku atau takut", "c": "menurut karena takut diancam",
      "d": "baru sadar setelah beberapa waktu"}
Q6 = {"a": "ada bukti digital", "b": "ada saksi", "c": "tidak ada bukti, hanya ingatan", "d": "belum tahu"}

MAP_Q2 = {"a": ["keamanan_diri"], "b": ["hak_perempuan"], "c": ["hak_perempuan"], "e": ["hak_perempuan"]}
MAP_Q3 = {"a": ["hukum_tpks"], "c": ["hak_perempuan"]}
MAP_Q4 = {"a": ["pelecehan_nonfisik"], "b": ["pelecehan_nonfisik"], "c": ["keamanan_diri"], "d": ["keamanan_diri"],
          "e": ["keamanan_diri"], "f": ["pelecehan_fisik"], "g": ["pemaksaan_hubungan"], "h": ["hukum_tpks"]}

# ---- Deteksi darurat dari pesan bebas -----------------------------------------
POLA_DARURAT = [
    # ancaman terhadap nyawa / kekerasan yang sedang terjadi
    r"\b(mau|akan|bakal|ingin)\s+(dibunuh|membunuh|bunuh\s+aku|bunuh\s+saya)",
    r"\bdiancam\s+(akan\s+)?(dibunuh|dibacok|ditusuk|ditembak)",
    r"\b(sedang|lagi|sekarang)\s+(dipukul|dipukuli|dicekik|disiksa|diperkosa|dikejar|diikuti|disekap)",
    r"\b(disekap|dikurung|dikunci\s+di)",
    r"\b(ada|datang|menunggu)\s+di\s+(depan|luar)\s+(rumah|kamar|kos|kosan)",
    r"\b(bawa|membawa)\s+(pisau|senjata|golok|parang)",
    # risiko menyakiti diri sendiri
    r"\b(bunuh\s+diri|mengakhiri\s+hidup|ingin\s+mati|mau\s+mati|menyakiti\s+diri|melukai\s+diri|tidak\s+ingin\s+hidup)",
]
POLA_DARURAT_RE = [re.compile(p) for p in POLA_DARURAT]

def deteksi_darurat(pesan):
    """Mengembalikan daftar pola darurat yang cocok di pesan (kosong jika aman)."""
    if not pesan:
        return []
    t = re.sub(r"[^a-z0-9\s]", " ", pesan.lower())
    t = re.sub(r"\s+", " ", t)
    return [p.pattern for p in POLA_DARURAT_RE if p.search(t)]

def gerbang(ans, pesan=None):
    # Gerbang Keselamatan: darurat jika Q1 = 'a' ATAU pesan bebas mengandung tanda bahaya
    if (ans or {}).get("q1") == "a":
        return "darurat"
    if deteksi_darurat(pesan):
        return "darurat"
    return "lanjut"

def rencana(ans):
    ans = ans or {}
    topics, types = [], []
    topics += MAP_Q2.get(ans.get("q2"), [])
    topics += MAP_Q3.get(ans.get("q3"), [])
    for x in ans.get("q4", []) or []:
        topics += MAP_Q4.get(x, [])
    if ans.get("q6") == "a":
        topics += ["keamanan_diri"]; types += ["prosedur"]
    if ans.get("q6") in ("c", "d"):
        types += ["panduan_dukungan"]
    if ans.get("q1") == "c" or ans.get("q5") in ("b", "d"):
        types += ["panduan_dukungan"]; topics += ["kesehatan_mental"]
    return {"topics": sorted(set(topics)), "prefer_types": sorted(set(types))}

def susun_query(pesan, ans):
    ans = ans or {}
    bagian = []
    if pesan:
        bagian.append("Pertanyaan: " + pesan.strip())
    if ans.get("q4"):
        bagian.append("Kejadian: " + "; ".join(Q4[x] for x in ans["q4"]) + ".")
    if ans.get("q2"):
        bagian.append("Lokasi: " + Q2[ans["q2"]] + ".")
    if ans.get("q3"):
        bagian.append("Pihak terlibat: " + Q3[ans["q3"]] + ".")
    if ans.get("q6"):
        bagian.append("Bukti: " + Q6[ans["q6"]] + ".")
    return " ".join(bagian)

def cari(index, query, plan, k=TOP_K):
    qv = index.emb.queries([query])[0]
    df = index.kb.copy()
    df["sim"] = index.matrix @ qv
    df = df[df["is_active"]]
    df["skor"] = (df["sim"]
                  + TOPIC_BOOST * df["topic"].isin(plan["topics"])
                  + TYPE_BOOST * df["entry_type"].isin(plan["prefer_types"]))
    df = df[df["sim"] >= MIN_SIM].sort_values("skor", ascending=False).head(k)
    return df[["entry_type", "title", "article_number", "topic", "source_url", "original_text", "sim", "skor"]]

### 8a. Contoh Pemakaian

In [20]:
# ---- Contoh pemakaian ---------------------------------------------------------
contoh_jawaban = {"q1": "b", "q2": "a", "q3": "d", "q4": ["c", "d"], "q5": "c", "q6": "a"}
contoh_pesan = "Aku diancam fotoku akan disebar, aku harus bagaimana?"

print("Gerbang:", gerbang(contoh_jawaban, contoh_pesan))
plan = rencana(contoh_jawaban)
query = susun_query(contoh_pesan, contoh_jawaban)
print("Rencana:", plan)
print("Query  :", query)
hasil = cari(idx, query, plan)
hasil[["title", "article_number", "topic", "sim", "skor"]]

Gerbang: lanjut
Rencana: {'topics': ['keamanan_diri'], 'prefer_types': ['prosedur']}
Query  : Pertanyaan: Aku diancam fotoku akan disebar, aku harus bagaimana? Kejadian: dikirimi foto atau video tanpa izin; diancam foto atau datanya disebar. Lokasi: Online (chat, medsos). Pihak terlibat: Orang yang baru dikenal. Bukti: ada bukti digital.


,title,article_number,topic,sim,skor
21,UU TPKS - Pasal 14 (bagian 1),14,keamanan_diri,0.833817,0.883817
78,UU TPKS - Pasal 57 (bagian 2),57,keamanan_diri,0.821504,0.871504
67,UU TPKS - Pasal 49 (bagian 2),49,keamanan_diri,0.821503,0.871503


### 8b. Uji Gerbang Keselamatan

In [21]:
uji = [
    ("Dia ada di depan rumahku sekarang",            "darurat"),
    ("Aku mau dibunuh sama pacarku",                 "darurat"),
    ("Aku lagi dipukuli suamiku",                    "darurat"),
    ("Aku ingin mati saja",                          "darurat"),
    ("Foto pribadiku diancam mau disebar",           "lanjut"),
    ("Apa hak korban pelecehan di tempat kerja?",    "lanjut"),
]
ans_aman = {"q1": "b"}
for pesan, harapan in uji:
    hasil = gerbang(ans_aman, pesan)
    print(("OK   " if hasil == harapan else "GAGAL"), "|", hasil, "|", pesan)

OK    | darurat | Dia ada di depan rumahku sekarang
OK    | darurat | Aku mau dibunuh sama pacarku
OK    | darurat | Aku lagi dipukuli suamiku
OK    | darurat | Aku ingin mati saja
OK    | lanjut | Foto pribadiku diancam mau disebar
OK    | lanjut | Apa hak korban pelecehan di tempat kerja?


## 9. Svarati AI (Gemini): Pedoman PFA & Cek Kutipan

In [24]:
GEMINI_MODEL = "gemini-3.5-flash"

SYSTEM_PROMPT = (
    "Kamu adalah Svarati AI, pendamping empati awal untuk penyintas kekerasan terhadap perempuan.\n"
    "Terapkan pedoman Psychological First Aid (PFA):\n"
    "1. Validasi emosi pengguna dengan hangat, tenang, tanpa menghakimi, dan tegaskan bahwa ini BUKAN kesalahan korban.\n"
    "2. Jawab pertanyaan HANYA berdasarkan informasi pada KONTEKS DOKUMEN.\n"
    "3. Jangan mengarang pasal, nomor telepon, atau alamat yang tidak ada di konteks.\n"
    "4. Hormati keputusan pengguna; berikan pilihan langkah berikutnya tanpa tekanan.\n"
    "5. Jika konteks tidak mencukupi, sampaikan keterbatasannya dengan jujur dan sarankan berkonsultasi ke layanan terverifikasi."
)



def bangun_prompt(pesan, hasil):
    sumber = "\n\n".join(
        f"[{i}] {r.title}" + (f" (Pasal {r.article_number})" if r.article_number else "") + f"\n{r.original_text}"
        for i, r in enumerate(hasil.itertuples(), 1))
    return f"SUMBER:\n{sumber}\n\nPERTANYAAN PENGGUNA:\n{pesan}"

def cek_kutipan(jawaban, hasil):
    # Pasal yang dikutip harus muncul di nomor pasal atau teks hasil retrieval
    dikutip = set(re.findall(r"Pasal\s+(\d+[A-Z]?)", jawaban))
    tersedia = set(hasil["article_number"].dropna().astype(str))
    for t in hasil["original_text"]:
        tersedia |= set(re.findall(r"Pasal\s+(\d+[A-Z]?)", t))
    return sorted(dikutip - tersedia)

def tanya_svarati(index, pesan, ans=None):
    if gerbang(ans, pesan) == "darurat":
        return {"mode": "darurat", "jawaban": "Hubungi 112 (Pusat Darurat) atau 129 (SAPA KemenPPPA) sekarang.", "sumber": None}
    plan, query = rencana(ans), susun_query(pesan, ans)
    hasil = cari(index, query, plan)
    if hasil.empty:
        return {"mode": "tanpa_sumber", "jawaban": "Aku belum menemukan sumber yang sesuai. Kamu bisa menghubungi pendamping atau layanan terverifikasi.", "sumber": hasil}
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        key = os.environ.get("GEMINI_API_KEY")
    if not key:
        return {"mode": "tanpa_llm", "jawaban": "(GEMINI_API_KEY belum diisi, hanya retrieval)", "sumber": hasil}
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    resp = client.models.generate_content(
        model=GEMINI_MODEL, contents=bangun_prompt(query, hasil),
        config=types.GenerateContentConfig(temperature=0.2, system_instruction=SYSTEM_PROMPT))
    jawaban = resp.text
    salah = cek_kutipan(jawaban, hasil)
    return {"mode": "rag", "jawaban": jawaban, "sumber": hasil, "pasal_tidak_ada_di_sumber": salah}

out = tanya_svarati(idx, contoh_pesan, contoh_jawaban)
print(out["mode"]); print(out["jawaban"]); print(out.get("pasal_tidak_ada_di_sumber"))

rag
Halo, aku mendengarkanmu. Aku memahami betapa takut, cemas, dan bingungnya perasaanmu saat menghadapi ancaman ini. Pertama-tama, tolong ketahuilah bahwa **ini sama sekali BUKAN kesalahanmu**. Kamu adalah korban dari tindakan yang tidak bertanggung jawab, dan kamu tidak sendirian.

Berdasarkan konteks hukum yang berlaku (UU TPKS), berikut adalah informasi hukum yang dapat menjadi peganganmu:

1. **Tindakan Pelaku Adalah Tindakan Pidana (Pasal 14)**
   Tindakan mengambil gambar/tangkapan layar, mengirimkan dokumen bermuatan seksual tanpa izin, serta menggunakannya untuk **pengancaman atau pemerasan** dikategorikan sebagai **Kekerasan Seksual Berbasis Elektronik**. Pelaku dapat dipidana dengan pidana penjara paling lama 4 (empat) tahun dan/atau denda paling banyak Rp200.000.000,00 (dua ratus juta rupiah).

2. **Privasi dan Identitasmu Dilindungi Hukum (Pasal 57)**
   Jika kasus ini diproses secara hukum, penuntut umum wajib menghindari pencantuman atau penyalinrekatan gambar, foto, at

## 10. Simulasi Skenario Uji

In [25]:
# Skenario 1: Pengguna Aman tapi Bingung Soal Foto Diancam Disebar
ans_test = {"q1": "b", "q2": "a", "q3": "d", "q4": ["c", "d"], "q5": "c", "q6": "a"}
msg_test = "Foto pribadiku diancam mau disebar. Apa yang bisa aku lakukan secara hukum dan bukti apa yang harus disimpan?"

try:
    res = tanya_svarati(idx, msg_test, ans_test)
    print("=== MODUS RESPONS ===:", res["mode"])
    print("\n=== JAWABAN AI ===:\n", res["jawaban"])
    if res.get("pasal_tidak_ada_di_sumber"):
        print("\n[PERATURAN SAFETY]: Terdeteksi pasal yang tidak ada di konteks:", res["pasal_tidak_ada_di_sumber"])
except Exception as e:
    print(f"Terjadi kesalahan saat pengujian: {e}")

=== MODUS RESPONS ===: rag

=== JAWABAN AI ===:
 Halo, terima kasih sudah berani bercerita. Saya memahami bahwa situasi ini pasti terasa sangat menakutkan, membuat cemas, dan membingungkan. Tolong ingat dan tanamkan dalam diri bahwa **ini sama sekali BUKAN kesalahanmu**. Kamu tidak bersalah atas tindakan pengancaman dan pelanggaran privasi yang dilakukan oleh orang lain.

Berdasarkan dokumen UU TPKS yang kamu berikan, berikut adalah penjelasan mengenai posisi hukum dari kejadian yang kamu alami:

### 1. Hal yang Bisa Dilakukan Secara Hukum (Berdasarkan UU TPKS)
* **Status Hukum Perbuatan Pelaku:** 
  Berdasarkan **Pasal 14 ayat (1) dan (2) UU TPKS**, tindakan merekam, mengambil tangkapan layar bermuatan seksual tanpa persetujuan, mentransmisikan dokumen elektronik bermuatan seksual, serta melakukan pengancaman atau pemerasan menggunakan hal tersebut dikategorikan sebagai **Kekerasan Seksual Berbasis Elektronik (KSBE)**.
* **Sanksi bagi Pelaku:** 
  Pelaku dapat dipidana dengan pidana p

## 11. Ringkasan Status Pipeline Awal

| Syarat Assignment | Status | Bukti di Notebook |
|---|---|---|
| Fitur final project dipilih | Terpenuhi | Svarati AI: pendamping empati dan rujukan pasal UU TPKS (bagian Deskripsi Fitur) |
| Pipeline awal dapat dijalankan | Terpenuhi | Ingestion, embedding 768 dimensi, dan retrieval berjalan . Generation Gemini [berhasil] |
| Output atau bukti pengujian | Terpenuhi | Contoh retrieval, uji gerbang keselamatan enam skenario (sel 8b) [isi hasil, misalnya: 6/6 lolos] |
| Hasil, keterbatasan, dan langkah lanjut | Terpenuhi | Lihat tabel Keterbatasan dan Langkah Lanjut di bawah |

### Keterbatasan

| No | Keterbatasan | Dampak |
|---|---|---|
| 1 | `DRIVE_SOURCES` baru berisi UU TPKS. Artikel edukasi (lima kategori) belum masuk index | Panduan langkah praktis belum bisa diambil; `TYPE_BOOST` belum berpengaruh |
| 2 | Integrasi ke backend belum dilakukan di notebook ini | Penyimpanan percakapan dan status validasi belum diuji |

### Langkah Pengembangan Berikutnya

| No | Langkah | Tujuan |
|---|---|---|
| 1 | Masukkan artikel edukasi ke `DRIVE_SOURCES` dengan `entry_type` yang sesuai | Index memuat panduan dan prosedur |
| 2 | Samakan nama topik artikel dengan `TOPIC_KEYWORDS` | Bonus topik bisa berlaku |
| 3 | Buat set pertanyaan uji beserta jawaban yang diharapkan | Mengukur kualitas retrieval |
| 4 | Integrasikan ke backend (`ai_messages`, `ai_message_sources`, `anonymous_sessions`) | Menguji penyimpanan percakapan dan status validasi |